# `01_langgraph_basic.ipynb`

`uv add langgraph langchain`

## Core Concept
1. `state` -> 노드들을 관통하는 데이터
1. Langgraph 기획 -> 노드/엣지 그림 + State에 어떤 데이터를 담을지를 기획하는 것

### 예시 - 여행 계획 세워주는 Workflow
1. wf가 채워야 하는 정보
    - 목적지: `str`
    - 여행일수: `int`
    - 목적지에서 방문할 장소들: `list[str]`
2. Node - Edge
    1. 목적지 채우는 Node
    2. 여행일수 Node
    3. 방문할 장소 채우는 Node
    4. 최종 답변을 만들어주는 LLM Node


In [3]:
from dotenv import load_dotenv
from typing_extensions import TypedDict, List
from langgraph.graph import StateGraph, START, END

load_dotenv()

True

In [18]:
# State 정의 -> 이름 잘짓기
class TripState(TypedDict):  # TypedDict? 딕셔너리(k-v)인데, Value의 자료형을 미리 정해놓은 dict
    message: str          # 사용자 입력메세지 - 시작과 동시에 채워질 데이터
    destination: str      # 목적지
    days: int             # 여행일수
    places: List[str]     # 방문장소목록 ex. ['경복궁', 'N타워', '한강']
    answer: str           # 최종 답변 - 종료시 완성될 메세지

In [19]:
# Node 정의 (함수)

def set_destination_node(state: TripState):
    import random
    pick = random.choice(['서울', '부산', '제주'])
    # Langgraph가 State 바뀐 부분만 넘기면, 자동으로 전체 + 갱신해서 넘김 (Partial Update 기능 지원)
    return {'destination': pick}


def set_days_node(state: TripState):
    return {'days': 3}


def collect_places_node(state: TripState):
    # state에서 정보를 뽑을것
    d = state['destination']

    if d == '서울':
        places = ['광화문', '국중박', '한강', 'N타워']
    elif d == '부산':
        places = ['해운대', '광안리', '서면', '기장']
    elif d == '제주':
        places = ['중문', '우도', '협재']

    return {'places': places}


def make_answer_node(state: TripState):
    destination = state['destination']
    days = state['days']
    places = state['places']

    answer = f'''{destination}을 추천합니다.
    {days}박 코스로 가시면 좋아요.
    {places}는 꼭 가보세요
    '''

    return {'answer': answer}

In [22]:
# builder 에서 Node들 연결 = 조립
builder = StateGraph(TripState)  # TripState 를 공유하는 Graph 빌더

# Node 등록 (이름, 노드(함수))
builder.add_node('set_destination_node', set_destination_node)
builder.add_node('collect_places_node', collect_places_node)
builder.add_node('set_days_node', set_days_node)
builder.add_node('make_answer_node', make_answer_node)

# Edge 연결 - 노드끼리 연결
builder.add_edge(START, 'set_destination_node')
builder.add_edge('set_destination_node', 'set_days_node')
builder.add_edge('set_days_node', 'collect_places_node')
builder.add_edge('collect_places_node', 'make_answer_node')
builder.add_edge('make_answer_node', END)

# graph 컴파일 (실행 가능하게 만든다)
graph = builder.compile()

In [24]:
# Langchain 세상에서 "실행" 은 .invoke() 가 기본 메서드
res_state = graph.invoke({  # Typed DICT니까, dict 를 시작시 넣기
    'message': '나 여행가고 싶어'  # 최초에 채울 k-v
})

print(res_state['answer'])

제주을 추천합니다.
    3박 코스로 가시면 좋아요.
    ['중문', '우도', '협재']는 꼭 가보세요
    
